In [ ]:
# %pip install -q pandas numpy rank-bm25 python-dotenv requests
# (Optional for embeddings)
# %pip install -q sentence-transformers faiss-cpu

from __future__ import annotations
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple
import random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

random.seed(7); np.random.seed(7)

In [ ]:
DATA_DIR = Path("data")

#### Load Data

In [ ]:
act_df = pd.read_csv(Path(DATA_DIR, 'activities.csv'))
print(act_df.head(5))

In [ ]:
# Activities by city
city_counts = act_df["city"].value_counts().sort_values(ascending=False).head(10)
plt.figure()
city_counts.plot(kind="bar")
plt.title("Top 10 Cities by # of Activities")
plt.xlabel("City"); plt.ylabel("Count")
plt.show()

In [ ]:
hotel_df = pd.read_csv(Path(DATA_DIR, 'hotels.csv'))
print(hotel_df.head(5))

In [ ]:
plt.figure()
hotel_df["nightly_price_usd"].plot(kind="hist", bins=20)
plt.title("Hotel Nightly Price Distribution")
plt.xlabel("USD"); plt.ylabel("Frequency")
plt.show()


In [ ]:
flight_df = pd.read_csv(Path(DATA_DIR, 'flights.csv'))
print(flight_df.head(5))

In [ ]:
plt.figure()
flight_df["on_time_rate"].plot(kind="hist", bins=20)
plt.title("Flight On-Time Rate Distribution")
plt.xlabel("Rate"); plt.ylabel("Frequency")
plt.show()

#### How to use Claude API

In [ ]:
import anthropic
client = anthropic.Anthropic()

system_prompt = "Being an expert in trip planning, write clear travel recommendations."
messages = []

def chat(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = logged_create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=system_prompt,
        messages=messages,
        temperature=0.2
    )
    reply = response.content[0].text
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("I need a 5 day travel plan from San Francisco to Los Angeles."))
print(chat("What about budget options for accommodation?"))  # Claude remembers the prior turn


Logging my request as Claude does not support viewing input

In [ ]:
def logged_create(**kwargs):
    print("=== INPUT ===")
    if "system" in kwargs:
        print(f"[system]: {kwargs['system']}\n")
    for msg in kwargs.get("messages", []):
        print(f"[{msg['role']}]: {msg['content']}\n")
    print("=============")

    response = client.messages.create(**kwargs)

    print("=== OUTPUT ===")
    print(response.content[0].text)
    print("==============\n")

    return response


#### RAG — Step 1: BM25 Keyword Retrieval

In [ ]:
from rank_bm25 import BM25Okapi

# --- Step 1: Serialize each row into a text document ---
# Each doc is a plain string of all field values; we keep a parallel list of
# the original row dicts so we can return structured data after retrieval.

def build_corpus(dataframes: dict) -> tuple[list[str], list[dict]]:
    docs, metadata = [], []
    for source, df in dataframes.items():
        for _, row in df.iterrows():
            # TODO1: join each row into lowercase cause BM25 treat upper and lower different
            text = " ".join(str(value) for value in row.values).lower()
            docs.append(text)

            # TODO2: store the source name + row data as a dict in metadata
            row_dict = row.to_dict()
            row_dict["source"] = source
            row_dict["text"] = text
            metadata.append(row_dict)

    return docs, metadata


dataframes = {"activities": act_df, "hotels": hotel_df, "flights": flight_df}
corpus_docs, corpus_meta = build_corpus(dataframes)
#print(corpus_docs)
#(corpus_meta)


In [ ]:
# --- Step 2: Build BM25 index ---
tokenized_corpus = [doc.split() for doc in corpus_docs]
bm25 = BM25Okapi(tokenized_corpus)

print(f"Index built: {len(corpus_docs)} documents")

In [ ]:
import re

# --- Tokenizer strategies ---
def tokenize_simple(text):
    """Basic whitespace split — mirrors build_corpus"""
    return text.lower().split()

def tokenize_clean(text):
    """Strip punctuation before splitting — symbols become spaces"""
    return re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()


# --- Text builder strategies (how a row becomes a document) ---
def text_all_fields(source, row):
    """Index every field — current default"""
    return " ".join(str(v) for v in row.values)

def text_selective(source, row):
    """Only index meaningful text fields — drop numeric/time fields"""
    if source == "activities":
        return f"{row['name']} {row['city']} {row['theme']} {row['notes']}"
    elif source == "hotels":
        return f"{row['name']} {row['city']} {row['neighborhood']} {row['notes']}"
    elif source == "flights":
        return f"{row['origin']} {row['destination']} {row['airline']}"

def text_weighted(source, row):
    """Repeat key fields to boost their BM25 term frequency"""
    if source == "activities":
        return f"{row['name']} {row['name']} {row['city']} {row['theme']} {row['theme']} {row['notes']}"
    elif source == "hotels":
        return f"{row['name']} {row['name']} {row['city']} {row['neighborhood']} {row['notes']}"
    elif source == "flights":
        return f"{row['origin']} {row['origin']} {row['destination']} {row['airline']}"


# --- Strategy presets: (text_builder, tokenizer) ---
STRATEGIES = {
    "default":   (text_all_fields, tokenize_simple),   # original behaviour
    "clean":     (text_all_fields, tokenize_clean),    # strip symbols
    "selective": (text_selective,  tokenize_simple),   # text fields only
    "weighted":  (text_weighted,   tokenize_simple),   # boost key fields
}


# --- Generic index builder ---
def build_index(dataframes, text_fn, tokenize_fn):
    docs, metadata = [], []
    for source, df in dataframes.items():
        for _, row in df.iterrows():
            text = text_fn(source, row).lower()
            tokens = tokenize_fn(text)
            docs.append(tokens)
            row_dict = row.to_dict()
            row_dict["source"] = source
            metadata.append(row_dict)
    return BM25Okapi(docs), metadata


# --- Generic retrieve ---
def retrieve(query, bm25_index, corpus_meta, tokenize_fn=tokenize_simple, top_k=5):
    tokens = tokenize_fn(query)
    scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [corpus_meta[i] for i in top_indices if scores[i] > 0]


# --- Compare all strategies side by side ---
def compare_strategies(query, top_k=5):
    print(f'Query: "{query}"\n')
    for name, (text_fn, tok_fn) in STRATEGIES.items():
        bm25_idx, meta = build_index(dataframes, text_fn, tok_fn)
        results = retrieve(query, bm25_idx, meta, tok_fn, top_k)
        print(f"  [{name}]")
        if not results:
            print("    (no matches)")
        for r in results:
            label = r.get("name") or f"{r.get('origin')} → {r.get('destination')}"
            print(f"    [{r['source']}] {label}")
        print()


In [ ]:
# --- Try different queries to see how strategies differ ---
compare_strategies("outdoor sports activities new york")
compare_strategies("cheap flights from San Francisco") #weighted for origin based flight queries
compare_strategies("08:00")   # interesting: clean is only one matched the time with pure number
compare_strategies("San Francisco at 08:00")


In [ ]:
# --- Step 4: Format retrieved results as a context string for Claude ---

def format_context(results: list[dict]) -> str:
    lines = []
    for r in results:
        source = r["source"]
        field_string = []
        fields = {k: v for k, v in r.items() if k != "source"}

        for key, value in fields.items():
            field_string.append(f"{key}: {value}")

        lines.append(f"[{source.upper()}] {' | '.join(field_string)}")
    return "\n".join(lines)



In [ ]:
# --- Step 5: RAG-augmented chat ---

default_bm25, default_meta = build_index(dataframes, text_all_fields, tokenize_simple)

rag_messages = []

def rag_chat(user_input: str, top_k: int = 5) -> str:
    results = retrieve(user_input, default_bm25, default_meta, tokenize_simple, top_k)
    context = format_context(results)

    content = (
        "Use the retrieved data below to answer the question.\n\n"
        f"Data:\n{context}\n\n"
        f"Question: {user_input}"
    )

    rag_messages.append({"role": "user", "content": content})
    response = logged_create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=system_prompt,
        messages=rag_messages,
        temperature=0.2
    )
    reply = response.content[0].text
    rag_messages.append({"role": "assistant", "content": reply})
    return reply


#### RAG Chat Queries

In [ ]:
# --- Single-turn queries ---
rag_messages.clear()

rag_chat("What are some sports activities in New York City?")
rag_chat("Find me a cheap flight from San Francisco to Los Angeles.")
rag_chat("What are the best rated hotels in Chicago?")

# --- Multi-turn: follow-up uses prior context ---
rag_messages.clear()

rag_chat("I want to visit New York City for outdoor activities.")
rag_chat("What hotels are nearby?")  # Claude remembers New York City, \
                                    # but BM25 does not know previous queries


## Next Plan: Retrieval Strategy Comparison

### Goal
Compare three retrieval strategies side by side to understand their trade-offs:

| Strategy | Approach | Strength | Weakness |
|---|---|---|---|
| **BM25** ✅ done | Term frequency + IDF scoring | Fast, no model needed | Exact token match only — `"cheap"` never matches `cost_usd: 40` |
| **Exact Keyword Match** ← next | `in` substring search | Handles partial matches e.g. `"08:00"` matches `"08:00-20:00"` | No ranking, no semantics |
| **Embeddings** | Semantic vector similarity | Understands meaning — `"cheap"` can match low `cost_usd` | Slower, needs a model |

---

### Next Step: Exact Keyword Match

Implement `retrieve_exact(query, corpus_meta, top_k)` that:
1. Splits query into individual keywords (lowercased)
2. For each doc in `corpus_meta`, counts how many keywords appear as **substrings** in the doc text
3. Ranks results by hit count (most matches first), returns top_k in the same `list[dict]` format as `retrieve()`

Then add `compare_retrieval(query)` that runs BM25 and exact keyword match on the same query side by side.

### Key questions to answer
- Does exact match fix the `"08:00"` vs `"08:00-20:00"` problem BM25 had with the `default` strategy?
- Does substring matching cause false positives (e.g. `"or"` matching `"New York"`)?
- Does it return more or fewer results than BM25 for the same query?

### Shared infrastructure already in place
- `corpus_meta` — list of row dicts with `"source"` key (reuse directly)
- `build_corpus()` — provides `corpus_docs` with the lowercased text strings (reuse as the search corpus)
- `format_context()` — formats results for Claude (no changes needed)
- `rag_chat()` — plugs in any retrieval function that returns `list[dict]`
- `logged_create()` — logs all Claude API calls automatically
